In [ ]:
# Week 1: Data exploration and preparation
import pandas as pd
from pathlib import Path
import ast
import re

DATA_DIR = Path("../data")
df = pd.read_csv(DATA_DIR / "recipes.csv")
df.head()

In [ ]:
# Keep fields used for matching, attribution and review-aware ranking.
useful_columns = [
    "RecipeId", "Name", "AuthorName", "TotalTime",
    "RecipeIngredientQuantities", "RecipeIngredientParts",
    "RecipeServings", "AggregatedRating", "ReviewCount"
]
df = df[useful_columns].dropna(subset=["RecipeIngredientParts"]).copy()
df.head()

In [ ]:
# Inspect the raw ingredient format before cleaning.
print(df.info())
print(df["RecipeIngredientParts"].head(10))

In [ ]:
def parse_ingredients(raw):
    """Parse Food.com R-style c("ingredient", ...) vectors without altering names."""
    if pd.isna(raw):
        return []
    text = str(raw).strip()
    if text.startswith("c(") and text.endswith(")"):
        text = text[2:-1].strip()
    if not text:
        return []
    # Quoted R vector values are compatible with Python's string literal syntax.
    try:
        parts = ast.literal_eval("[" + text + "]")
    except (ValueError, SyntaxError):
        # Fallback for unusual rows; preserve the ingredient's first character.
        parts = [part.strip().strip('"').strip("'") for part in text.split(",")]
    return [re.sub(r"\\s+", " ", str(item)).strip().lower()
            for item in parts if str(item).strip()]

# Store a readable, clean string for exploration; the original column is retained.
df["cleaned_ingredients"] = df["RecipeIngredientParts"].apply(
    lambda raw: ", ".join(parse_ingredients(raw))
)
df[["RecipeIngredientParts", "cleaned_ingredients"]].head(10)

In [ ]:
# Sanity checks: legitimate c-initial ingredients must remain intact.
assert parse_ingredients('c("cinnamon", "chives", "cream")') == [
    "cinnamon", "chives", "cream"
]
assert parse_ingredients('c("blueberries", "granulated sugar")') == [
    "blueberries", "granulated sugar"
]
assert parse_ingredients(None) == []
print("Ingredient parser checks passed.")

In [ ]:
# Save the source ingredient column too: the matching notebook parses it directly.
DATA_DIR.mkdir(exist_ok=True)
df.to_csv(DATA_DIR / "cleaned_recipes.csv", index=False)
print("Saved", DATA_DIR / "cleaned_recipes.csv")